In [1]:
import torch
import sys

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected. Check your Colab runtime configuration.")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [1]:
# Summary of the MSRIIndic_JNUTagset Dataset

from collections import Counter
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------------
# 1. Reliable Dataset Search (Local Laptop Path + Auto-Detection)
# ------------------------------------------------------------------
csv_candidates = [
    Path(
        "/home/badshah/Desktop/Cross-Domain-23/data/raw/Literature_Corpus.csv"
    ),
    Path.home()
    / "Desktop"
    / "Cross-Domain-23"
    / "data"
    / "raw"
    / "Literature_Corpus.csv",
    Path("data/raw/Literature_Corpus.csv"),
    Path("../data/raw/Literature_Corpus.csv"),
    Path.cwd() / "data" / "raw" / "Literature_Corpus.csv",
    Path.cwd().parent / "data" / "raw" / "Literature_Corpus.csv",
    Path("/content/data/raw/Literature_Corpus.csv"),
]

csv_path = next((path.resolve() for path in csv_candidates if path.is_file()), None)

# Upward search fallback if launched from an unexpected directory
if csv_path is None:
  curr = Path.cwd().resolve()
  for parent in [curr, *curr.parents]:
    target = parent / "data" / "raw" / "Literature_Corpus.csv"
    if target.is_file():
      csv_path = target
      break

if csv_path is None:
  raise FileNotFoundError(
      "Literature_Corpus.csv was not found. Please check the path!"
  )

print(f"Dataset: {csv_path}\n")

# ------------------------------------------------------------------
# 2. Read Dataset & Clean Columns
# ------------------------------------------------------------------
dataset = pd.read_csv(csv_path, encoding="utf-8-sig")
required_columns = {"sentence_id", "word", "tag"}
missing_columns = required_columns.difference(dataset.columns)
if missing_columns:
  raise ValueError(
      f"Dataset is missing required columns: {sorted(missing_columns)}"
  )

words = dataset["word"].dropna().astype(str).str.strip()
nonempty_words = words[words.ne("")]

tags = dataset["tag"].dropna().astype(str).str.strip()
nonempty_tags = tags[tags.ne("")]

total_tokens = len(dataset)
total_sentences = dataset["sentence_id"].nunique(dropna=True)
unique_words = nonempty_words.nunique()
unique_tags = nonempty_tags.nunique()

# ------------------------------------------------------------------
# 3. Core Metrics Output (Exact Labels)
# ------------------------------------------------------------------
print("=" * 65)
print("Literature Corpus Dataset Summary")
print("=" * 65)
print(f"Total Tokens: {total_tokens:,}")
print(f"Total Sentences: {total_sentences:,}")
print(f"Unique Words: {unique_words:,}")
print(f"Unique POS Tags: {unique_tags}")
print(
    f"Tagged Tokens: {len(nonempty_tags):,}"
    f" ({len(nonempty_tags)/total_tokens*100:.2f}%)"
)
print(
    f"Untagged Tokens: {total_tokens - len(nonempty_tags)}"
    f" ({(total_tokens - len(nonempty_tags))/total_tokens*100:.2f}%)"
)
print(
    "Average Sentence Length:"
    f" {total_tokens/total_sentences:.2f} tokens/sentence"
)
print("=" * 65)

# ------------------------------------------------------------------
# 4. Full POS Tag Distribution (All 28 Tags Ranked by Frequency)
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("Token count by tag (Ranked by Frequency):")
print("=" * 65)
print(" Rank   Tag       Token Count   Percentage   Cumulative %")
print("-" * 65)
cum_pct = 0.0
tag_counts = nonempty_tags.value_counts()
for rank, (t, c) in enumerate(tag_counts.items(), start=1):
  pct = c / len(nonempty_tags) * 100
  cum_pct += pct
  print(f"  {rank:2d}.   {t:<8}  {c:7,d}       {pct:6.2f}%       {cum_pct:6.2f}%")
print("-" * 65)

# ------------------------------------------------------------------
# 5. Top 20 Most Frequent Words
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("Top 20 Most Frequent Tokens:")
print("=" * 65)
print(" Rank   Token            Frequency    Percentage")
print("-" * 65)
word_counts = nonempty_words.value_counts().head(20)
for rank, (w, c) in enumerate(word_counts.items(), start=1):
  pct = c / total_tokens * 100
  print(f"  {rank:2d}.   {w:<16} {c:6,d}       {pct:5.2f}%")
print("-" * 65)

Dataset: /home/siba/P25DS023/Cross-Domain-23/data/raw/Literature_Corpus.csv

Literature Corpus Dataset Summary
Total Tokens: 11,899
Total Sentences: 1,461
Unique Words: 4,518
Unique POS Tags: 560
Tagged Tokens: 11,875 (99.80%)
Untagged Tokens: 24 (0.20%)
Average Sentence Length: 8.14 tokens/sentence

Token count by tag (Ranked by Frequency):
 Rank   Tag       Token Count   Percentage   Cumulative %
-----------------------------------------------------------------
   1.   PU          2,194        18.48%        18.48%
   2.   CX            827         6.96%        25.44%
   3.   ALC           347         2.92%        28.36%
   4.   V.sg.3.prs.n      346         2.91%        31.28%
   5.   NC.mas.sg.0.i      282         2.37%        33.65%
   6.   CEM           280         2.36%        36.01%
   7.   V.sg.3.imprf.n      256         2.16%        38.16%
   8.   AMN           251         2.11%        40.28%
   9.   VBP.mas.sg.0.i      235         1.98%        42.26%
  10.   NC.neu.sg.acc.ii 

In [2]:
# Extract All Unique POS Tags from Literature Corpus Dataset

from pathlib import Path
import pandas as pd

# -------------------------------------------------------------
# 1. Load Literature Corpus Dataset
# -------------------------------------------------------------
csv_candidates = [
    Path("/home/badshah/Desktop/Cross-Domain-23/data/raw/Literature_Corpus.csv"),
    Path.home() / "Desktop" / "Cross-Domain-23" / "data" / "raw" / "Literature_Corpus.csv",
    Path("data/raw/Literature_Corpus.csv"),
    Path("../data/raw/Literature_Corpus.csv"),
    Path.cwd() / "data" / "raw" / "Literature_Corpus.csv",
    Path.cwd().parent / "data" / "raw" / "Literature_Corpus.csv",
    Path("/content/data/raw/Literature_Corpus.csv"),
]
csv_path = next((path.resolve() for path in csv_candidates if path.is_file()), None)

if csv_path is None:
    raise FileNotFoundError("Literature_Corpus.csv was not found.")

dataset = pd.read_csv(csv_path, encoding="utf-8-sig")

# -------------------------------------------------------------
# 2. Extract and Sort Unique POS Tags
# -------------------------------------------------------------
tags = dataset["tag"].dropna().astype(str).str.strip()
tags = tags[tags.ne("")]
unique_tags = sorted(tags.unique().tolist())

print(f"Total Unique Tags ({len(unique_tags)}):\n")
print(unique_tags)

# -------------------------------------------------------------
# 3. Print Formatted List with Token Counts (Top 40 + Summary)
# -------------------------------------------------------------
print("\n" + "=" * 55)
print(f"  TOP 40 MOST FREQUENT TAGS (OUT OF {len(unique_tags)} TOTAL)")
print("=" * 55)
tag_counts = tags.value_counts()
for rank, (tag, count) in enumerate(tag_counts.head(40).items(), start=1):
    pct = (count / len(tags)) * 100
    print(f"  {rank:2d}. {tag:<25} : {count:6,d} tokens ({pct:5.2f}%)")
print("=" * 55)
print(f"  ... and {len(unique_tags) - 40} additional fine-grained tags.")
print("=" * 55)

Total Unique Tags (560):

['ALC', 'ALC.0', 'ALC.loc', 'AMN', 'AMN.0', 'AMN.ins', 'CCD', 'CCD.n', 'CCL', 'CCL.n', 'CEM', 'CIN', 'CNG', 'CSB', 'CSB.n', 'CX', 'DAB.0.sg.1.0.i.prx.n', 'DAB.fem.pl.3.gen.vi.prx.n', 'DAB.fem.sg.3.0.i.dst.n', 'DAB.fem.sg.3.0.i.prx.n', 'DAB.fem.sg.3.ins.iii.dst.n', 'DAB.fem.sg.3.ins.iii.prx.n', 'DAB.fem.sg.3.loc.vii.dst.n', 'DAB.fem.sg.3.loc.vii.prx.n', 'DAB.fem.sg.3.nom.i.dst.n', 'DAB.fem.sg.3.nom.i.prx.n', 'DAB.mas.du.3.nom.i.prx.n', 'DAB.mas.pl.3.gen.vi.dst.n', 'DAB.mas.pl.3.gen.vi.prx.n', 'DAB.mas.pl.3.ins.iii.dst.n', 'DAB.mas.pl.3.loc.vii.dst.n', 'DAB.mas.pl.3.loc.vii.prx.n', 'DAB.mas.pl.3.nom.i.dst.n', 'DAB.mas.pl.3.nom.i.prx.n', 'DAB.mas.sg.3.0.i.dst.n', 'DAB.mas.sg.3.0.i.prx.n', 'DAB.mas.sg.3.abl.v.dst.n', 'DAB.mas.sg.3.acc.ii.dst.n', 'DAB.mas.sg.3.acc.ii.prx.n', 'DAB.mas.sg.3.dat.iv.dst.n', 'DAB.mas.sg.3.gen.vi.dst.n', 'DAB.mas.sg.3.gen.vi.prx.n', 'DAB.mas.sg.3.ins.iii.dst.n', 'DAB.mas.sg.3.ins.iii.prx.n', 'DAB.mas.sg.3.loc.vii.dst.n', 'DAB.mas.sg.3.lo

In [3]:
# Summary of the Panchatantra Dataset

from collections import Counter
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------------
# 1. Reliable Dataset Search (Local Laptop Path + Auto-Detection)
# ------------------------------------------------------------------
csv_candidates = [
    Path(
        "/home/badshah/Desktop/Cross-Domain-23/data/raw/Panchatantra_clean.csv"
    ),
    Path.home()
    / "Desktop"
    / "Cross-Domain-23"
    / "data"
    / "raw"
    / "Panchatantra_clean.csv",
    Path("data/raw/Panchatantra_clean.csv"),
    Path("../data/raw/Panchatantra_clean.csv"),
    Path.cwd() / "data" / "raw" / "Panchatantra_clean.csv",
    Path.cwd().parent / "data" / "raw" / "Panchatantra_clean.csv",
    Path("/content/data/raw/Panchatantra_clean.csv"),
]

csv_path = next((path.resolve() for path in csv_candidates if path.is_file()), None)

# Upward search fallback if launched from an unexpected directory
if csv_path is None:
  curr = Path.cwd().resolve()
  for parent in [curr, *curr.parents]:
    target = parent / "data" / "raw" / "Panchatantra_clean.csv"
    if target.is_file():
      csv_path = target
      break

if csv_path is None:
  raise FileNotFoundError(
      "Panchatantra_clean.csv was not found. Please check the path!"
  )

print(f"Dataset: {csv_path}\n")

# ------------------------------------------------------------------
# 2. Read Dataset & Clean Columns
# ------------------------------------------------------------------
dataset = pd.read_csv(csv_path, encoding="utf-8-sig")
required_columns = {"sentence_id", "word", "tag"}
missing_columns = required_columns.difference(dataset.columns)
if missing_columns:
  raise ValueError(
      f"Dataset is missing required columns: {sorted(missing_columns)}"
  )

words = dataset["word"].dropna().astype(str).str.strip()
nonempty_words = words[words.ne("")]

tags = dataset["tag"].dropna().astype(str).str.strip()
nonempty_tags = tags[tags.ne("")]

total_tokens = len(dataset)
total_sentences = dataset["sentence_id"].nunique(dropna=True)
unique_words = nonempty_words.nunique()
unique_tags = nonempty_tags.nunique()

# ------------------------------------------------------------------
# 3. Core Metrics Output (Exact Labels)
# ------------------------------------------------------------------
print("=" * 65)
print("Panchatantra Dataset Summary")
print("=" * 65)
print(f"Total Tokens: {total_tokens:,}")
print(f"Total Sentences: {total_sentences:,}")
print(f"Unique Words: {unique_words:,}")
print(f"Unique POS Tags: {unique_tags}")
print(
    f"Tagged Tokens: {len(nonempty_tags):,}"
    f" ({len(nonempty_tags)/total_tokens*100:.2f}%)"
)
print(
    f"Untagged Tokens: {total_tokens - len(nonempty_tags)}"
    f" ({(total_tokens - len(nonempty_tags))/total_tokens*100:.2f}%)"
)
print(
    "Average Sentence Length:"
    f" {total_tokens/total_sentences:.2f} tokens/sentence"
)
print("=" * 65)

# ------------------------------------------------------------------
# 4. Full POS Tag Distribution (All 28 Tags Ranked by Frequency)
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("Token count by tag (Ranked by Frequency):")
print("=" * 65)
print(" Rank   Tag       Token Count   Percentage   Cumulative %")
print("-" * 65)
cum_pct = 0.0
tag_counts = nonempty_tags.value_counts()
for rank, (t, c) in enumerate(tag_counts.items(), start=1):
  pct = c / len(nonempty_tags) * 100
  cum_pct += pct
  print(f"  {rank:2d}.   {t:<8}  {c:7,d}       {pct:6.2f}%       {cum_pct:6.2f}%")
print("-" * 65)

# ------------------------------------------------------------------
# 5. Top 20 Most Frequent Words
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("Top 20 Most Frequent Tokens:")
print("=" * 65)
print(" Rank   Token            Frequency    Percentage")
print("-" * 65)
word_counts = nonempty_words.value_counts().head(20)
for rank, (w, c) in enumerate(word_counts.items(), start=1):
  pct = c / total_tokens * 100
  print(f"  {rank:2d}.   {w:<16} {c:6,d}       {pct:5.2f}%")
print("-" * 65)

Dataset: /home/siba/P25DS023/Cross-Domain-23/data/raw/Panchatantra_clean.csv

Panchatantra Dataset Summary
Total Tokens: 43,653
Total Sentences: 5,393
Unique Words: 11,236
Unique POS Tags: 28
Tagged Tokens: 43,647 (99.99%)
Untagged Tokens: 6 (0.01%)
Average Sentence Length: 8.09 tokens/sentence

Token count by tag (Ranked by Frequency):
 Rank   Tag       Token Count   Percentage   Cumulative %
-----------------------------------------------------------------
   1.   NP          9,690        22.20%        22.20%
   2.   PUNC        9,413        21.57%        43.77%
   3.   SN          4,768        10.92%        54.69%
   4.   A           4,334         9.93%        64.62%
   5.   KP          3,854         8.83%        73.45%
   6.   KN          2,225         5.10%        78.55%
   7.   NV          1,958         4.49%        83.03%
   8.   KKS         1,317         3.02%        86.05%
   9.   AVK           876         2.01%        88.06%
  10.   AN            846         1.94%        90.0

In [4]:
# Extract All Unique POS Tags from Panchatantra Dataset

from pathlib import Path
import pandas as pd

# -------------------------------------------------------------
# 1. Load Panchatantra Dataset
# -------------------------------------------------------------
csv_candidates = [
    Path("/home/badshah/Desktop/Cross-Domain-23/data/raw/Panchatantra_clean.csv"),
    Path.home() / "Desktop" / "Cross-Domain-23" / "data" / "raw" / "Panchatantra_clean.csv",
    Path("data/raw/Panchatantra_clean.csv"),
    Path("../data/raw/Panchatantra_clean.csv"),
    Path.cwd() / "data" / "raw" / "Panchatantra_clean.csv",
    Path.cwd().parent / "data" / "raw" / "Panchatantra_clean.csv",
    Path("/content/data/raw/Panchatantra_clean.csv"),
]
csv_path = next((path.resolve() for path in csv_candidates if path.is_file()), None)

if csv_path is None:
    raise FileNotFoundError("Panchatantra_clean.csv was not found.")

dataset = pd.read_csv(csv_path, encoding="utf-8-sig")

# -------------------------------------------------------------
# 2. Extract and Sort Unique POS Tags
# -------------------------------------------------------------
tags = dataset["tag"].dropna().astype(str).str.strip()
tags = tags[tags.ne("")]
unique_tags = sorted(tags.unique().tolist())

print(f"Total Unique Tags ({len(unique_tags)}):\n")
print(unique_tags)

# -------------------------------------------------------------
# 3. Print Formatted List with Token Counts
# -------------------------------------------------------------
print("\n" + "=" * 45)
print(f"     ALL {len(unique_tags)} UNIQUE TAGS WITH FREQUENCY")
print("=" * 45)
for rank, (tag, count) in enumerate(tags.value_counts().items(), start=1):
    print(f"  {rank:2d}. {tag:<6} : {count:6,d} tokens")
print("=" * 45)


Total Unique Tags (28):

['A', 'ABS', 'AN', 'AP', 'APY', 'ASD', 'ASMB', 'AUD', 'AUP', 'AVK', 'DV', 'INTF', 'KKS', 'KN', 'KNV', 'KP', 'KV', 'NP', 'NS', 'NV', 'PUNC', 'SMK', 'SMKP', 'SMKY', 'SN', 'SNV', 'SSN', 'UA']

     ALL 28 UNIQUE TAGS WITH FREQUENCY
   1. NP     :  9,690 tokens
   2. PUNC   :  9,413 tokens
   3. SN     :  4,768 tokens
   4. A      :  4,334 tokens
   5. KP     :  3,854 tokens
   6. KN     :  2,225 tokens
   7. NV     :  1,958 tokens
   8. KKS    :  1,317 tokens
   9. AVK    :    876 tokens
  10. AN     :    846 tokens
  11. NS     :    729 tokens
  12. APY    :    541 tokens
  13. KV     :    401 tokens
  14. SNV    :    386 tokens
  15. AP     :    365 tokens
  16. AUD    :    348 tokens
  17. INTF   :    305 tokens
  18. KNV    :    278 tokens
  19. ASMB   :    213 tokens
  20. SSN    :    171 tokens
  21. ASD    :    133 tokens
  22. AUP    :    117 tokens
  23. SMKY   :     97 tokens
  24. ABS    :     87 tokens
  25. DV     :     85 tokens
  26. SMKP   :     56

In [5]:
# Summary of the News_Domain Dataset (News_Annotated.csv)

from collections import Counter
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------------
# 1. Reliable Dataset Search
# ------------------------------------------------------------------
csv_candidates = [
    Path("/home/badshah/Desktop/Cross-Domain-23/data/raw/News_Annotated.csv"),
    Path.home() / "Desktop" / "Cross-Domain-23" / "data" / "raw" / "News_Annotated.csv",
    Path("data/raw/News_Annotated.csv"),
    Path("../data/raw/News_Annotated.csv"),
    Path.cwd() / "data" / "raw" / "News_Annotated.csv",
    Path.cwd().parent / "data" / "raw" / "News_Annotated.csv",
    Path("/content/data/raw/News_Annotated.csv"),
]

csv_path = next((path.resolve() for path in csv_candidates if path.is_file()), None)

if csv_path is None:
    curr = Path.cwd().resolve()
    for parent in [curr, *curr.parents]:
        target = parent / "data" / "raw" / "News_Annotated.csv"
        if target.is_file():
            csv_path = target
            break

if csv_path is None:
    raise FileNotFoundError("News_Annotated.csv was not found. Please check the path!")

print(f"Dataset: {csv_path}\n")

# ------------------------------------------------------------------
# 2. Read Dataset & Automatically Adapt Column Names
# ------------------------------------------------------------------
dataset = pd.read_csv(csv_path, encoding="utf-8-sig")

# In News_Annotated.csv:
#   - word column is named "token"
#   - tag column is named "gold_sanskrit_tag"
#   - sentence identifier is "annotation_id" (doc_id + sentence_id)
word_col = "token" if "token" in dataset.columns else "word"
tag_col = "gold_sanskrit_tag" if "gold_sanskrit_tag" in dataset.columns else "tag"
sent_id_col = "annotation_id" if "annotation_id" in dataset.columns else "sentence_id"

words = dataset[word_col].dropna().astype(str).str.strip()
nonempty_words = words[words.ne("")]

tags = dataset[tag_col].dropna().astype(str).str.strip()
nonempty_tags = tags[tags.ne("")]

total_tokens = len(dataset)
total_sentences = dataset[sent_id_col].nunique(dropna=True)
unique_words = nonempty_words.nunique()
unique_tags = nonempty_tags.nunique()

# ------------------------------------------------------------------
# 3. Core Metrics Output
# ------------------------------------------------------------------
print("=" * 65)
print("News Annotated Dataset Summary")
print("=" * 65)
print(f"Total Tokens: {total_tokens:,}")
print(f"Total Sentences (Unique annotation_id): {total_sentences:,}")
print(f"Unique Words: {unique_words:,}")
print(f"Unique POS Tags ({tag_col}): {unique_tags}")
print(f"Tagged Tokens: {len(nonempty_tags):,} ({len(nonempty_tags)/total_tokens*100:.2f}%)")
print(f"Untagged Tokens: {total_tokens - len(nonempty_tags)} ({(total_tokens - len(nonempty_tags))/total_tokens*100:.2f}%)")
print(f"Average Sentence Length: {total_tokens/total_sentences:.2f} tokens/sentence")
print("=" * 65)

# ------------------------------------------------------------------
# 4. Full POS Tag Distribution (All 24 Tags Ranked by Frequency)
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print(f"Token count by tag ({tag_col} - Ranked by Frequency):")
print("=" * 65)
print(" Rank   Tag       Token Count   Percentage   Cumulative %")
print("-" * 65)
cum_pct = 0.0
tag_counts = nonempty_tags.value_counts()
for rank, (t, c) in enumerate(tag_counts.items(), start=1):
    pct = c / len(nonempty_tags) * 100
    cum_pct += pct
    print(f"  {rank:2d}.   {t:<8}  {c:7,d}       {pct:6.2f}%       {cum_pct:6.2f}%")
print("-" * 65)

# ------------------------------------------------------------------
# 5. Top 20 Most Frequent Words
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("Top 20 Most Frequent Tokens:")
print("=" * 65)
print(" Rank   Token            Frequency    Percentage")
print("-" * 65)
word_counts = nonempty_words.value_counts().head(20)
for rank, (w, c) in enumerate(word_counts.items(), start=1):
    pct = c / total_tokens * 100
    print(f"  {rank:2d}.   {w:<16} {c:6,d}       {pct:5.2f}%")
print("-" * 65)

Dataset: /home/siba/P25DS023/Cross-Domain-23/data/raw/News_Annotated.csv

News Annotated Dataset Summary
Total Tokens: 24,299
Total Sentences (Unique annotation_id): 2,645
Unique Words: 15,327
Unique POS Tags (gold_sanskrit_tag): 24
Tagged Tokens: 24,299 (100.00%)
Untagged Tokens: 0 (0.00%)
Average Sentence Length: 9.19 tokens/sentence

Token count by tag (gold_sanskrit_tag - Ranked by Frequency):
 Rank   Tag       Token Count   Percentage   Cumulative %
-----------------------------------------------------------------
   1.   N          12,108        49.83%        49.83%
   2.   NV          3,035        12.49%        62.32%
   3.   AVN         2,403         9.89%        72.21%
   4.   KP          2,197         9.04%        81.25%
   5.   NS          1,344         5.53%        86.78%
   6.   SN          1,180         4.86%        91.64%
   7.   AB            458         1.88%        93.52%
   8.   AVC           412         1.70%        95.22%
   9.   A             379         1.56%    

In [2]:
from pathlib import Path
import pandas as pd

# Load News_Annotated dataset
csv_path = Path("data/raw/News_Annotated.csv")
if not csv_path.is_file():
  csv_path = Path(
      "/home/badshah/Desktop/Cross-Domain-23/data/raw/News_Annotated.csv"
  )

dataset = pd.read_csv(csv_path, encoding="utf-8-sig")

# -------------------------------------------------------------
# 1. Print as a standard Python List
# -------------------------------------------------------------
unique_tags = sorted(dataset["gold_sanskrit_tag"].dropna().unique().tolist())

print(f"Total Unique Tags ({len(unique_tags)}):\n")
print(unique_tags)

# -------------------------------------------------------------
# 2. Print Formatted List with Token Counts
# -------------------------------------------------------------
print("\n" + "=" * 45)
print("     ALL 24 UNIQUE TAGS WITH FREQUENCY")
print("=" * 45)
for rank, (tag, count) in enumerate(
    dataset["gold_sanskrit_tag"].value_counts().items(), start=1
):
  print(f"  {rank:2d}. {tag:<6} : {count:6,d} tokens")
print("=" * 45)

FileNotFoundError: [Errno 2] No such file or directory: '/home/badshah/Desktop/Cross-Domain-23/data/raw/News_Annotated.csv'

In [1]:
# Summary of the News_Unannotated Dataset

from collections import Counter
from pathlib import Path
import re
import pandas as pd

# ------------------------------------------------------------------
# 1. Reliable Dataset Search (Local Laptop Path + Auto-Detection)
# ------------------------------------------------------------------
csv_candidates = [
    Path("/home/badshah/Desktop/Cross-Domain-23/data/raw/News_Unannotated.csv"),
    Path.home()
    / "Desktop"
    / "Cross-Domain-23"
    / "data"
    / "raw"
    / "News_Unannotated.csv",
    Path("data/raw/News_Unannotated.csv"),
    Path("../data/raw/News_Unannotated.csv"),
    Path.cwd() / "data" / "raw" / "News_Unannotated.csv",
    Path.cwd().parent / "data" / "raw" / "News_Unannotated.csv",
    Path("/content/data/raw/News_Unannotated.csv"),
]

csv_path = next((path.resolve() for path in csv_candidates if path.is_file()), None)

if csv_path is None:
  curr = Path.cwd().resolve()
  for parent in [curr, *curr.parents]:
    target = parent / "data" / "raw" / "News_Unannotated.csv"
    if target.is_file():
      csv_path = target
      break

if csv_path is None:
  raise FileNotFoundError(
      "News_Unannotated.csv was not found. Please check the path!"
  )

print(f"Dataset: {csv_path}\n")

# ------------------------------------------------------------------
# 2. Read Dataset (Sentence-Level Raw Corpus)
# ------------------------------------------------------------------
dataset = pd.read_csv(csv_path, encoding="utf-8-sig")

total_sentences = len(dataset)
unique_annotation_ids = (
    dataset["annotation_id"].nunique(dropna=True)
    if "annotation_id" in dataset.columns
    else total_sentences
)
unique_docs = (
    dataset["doc_id"].nunique(dropna=True)
    if "doc_id" in dataset.columns
    else "N/A"
)
distinct_sentences = (
    dataset["sentence"].dropna().str.strip().nunique()
    if "sentence" in dataset.columns
    else total_sentences
)

# Tokenize sentences
sentences_clean = dataset["sentence"].dropna().astype(str).str.strip()
ws_tokens = [w for s in sentences_clean for w in s.split() if w.strip()]

token_pattern = re.compile(r"[\w\u0900-\u097F]+|[^\w\s\u0900-\u097F]")
punct_tokens = []
for s in sentences_clean:
  punct_tokens.extend(token_pattern.findall(s))
word_counts = Counter(punct_tokens)
unique_words = len(word_counts)
total_tokens = len(punct_tokens)

# ------------------------------------------------------------------
# 3. Core Metrics Output
# ------------------------------------------------------------------
print("=" * 65)
print("News Unannotated Dataset Summary (Sentence-Level Corpus)")
print("=" * 65)
print(f"Total Sentences (Rows in CSV)       : {total_sentences:,}")
print(f"Distinct Sentence Texts             : {distinct_sentences:,}")
print(f"Total Source Articles (doc_id)      : {unique_docs:,}")
print(f"Total Tokens (Standard Tokenized)   : {total_tokens:,}")
print(f"Total Tokens (Whitespace Tokenized) : {len(ws_tokens):,}")
print(f"Unique Words (Vocabulary)           : {unique_words:,}")
print("Unique POS Tags                     : 0 (Unannotated Dataset)")
print(
    "Average Sentence Length             :"
    f" {total_tokens/total_sentences:.2f} tokens/sentence"
)
print("=" * 65)

# ------------------------------------------------------------------
# 4. Top 20 Most Frequent Words
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("Top 20 Most Frequent Tokens:")
print("=" * 65)
print(" Rank   Token            Frequency    Percentage")
print("-" * 65)
for rank, (w, c) in enumerate(word_counts.most_common(20), start=1):
  pct = c / total_tokens * 100
  print(f"  {rank:2d}.   {w:<16} {c:6,d}       {pct:5.2f}%")
print("-" * 65)

# ------------------------------------------------------------------
# 5. Sample Sanskrit News Sentences
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("Sample Sanskrit News Sentences:")
print("=" * 65)
for i in range(min(3, len(sentences_clean))):
  print(f"[{i+1}] {sentences_clean.iloc[i]}")
print("=" * 65)

Dataset: /home/siba/P25DS023/Cross-Domain-23/data/raw/News_Unannotated.csv

News Unannotated Dataset Summary (Sentence-Level Corpus)
Total Sentences (Rows in CSV)       : 3,000
Distinct Sentence Texts             : 2,977
Total Source Articles (doc_id)      : 1,554
Total Tokens (Standard Tokenized)   : 38,745
Total Tokens (Whitespace Tokenized) : 29,872
Unique Words (Vocabulary)           : 17,745
Unique POS Tags                     : 0 (Unannotated Dataset)
Average Sentence Length             : 12.91 tokens/sentence

Top 20 Most Frequent Tokens:
 Rank   Token            Frequency    Percentage
-----------------------------------------------------------------
   1.   -                 3,159        8.15%
   2.   ,                   672        1.73%
   3.   :                   581        1.50%
   4.   ‌                   490        1.26%
   5.   च                   392        1.01%
   6.   इति                 344        0.89%
   7.   ।                   335        0.86%
   8.   .         